In [14]:
import pandas as pd

In [15]:
columns = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
           'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
           'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
           'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
           'tunnel_parents', 'ip_proto']

In [16]:
df = pd.read_csv("../data/raw/zeek-captures/beacon_conn.log",sep='\t',comment='#', names = columns)

In [17]:
df['conn_state']

0     OTH
1     OTH
2      S0
3      S0
4     SHR
     ... 
80    OTH
81    SHR
82    SHR
83    SHR
84    OTH
Name: conn_state, Length: 85, dtype: object

In [18]:
print(df.shape)
df.head()

(85, 22)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,1.790988e+09,CxdbMj4U97n1WAxnm7,fe80::20c:29ff:fe6c:6d6c,5353,ff02::fb,5353,udp,-,-,-,...,T,F,0,C,0,0,0,0,-,17
1,1.790988e+09,CppJk51WOITOEcSk94,192.168.150.129,5353,224.0.0.251,5353,udp,-,-,-,...,T,T,0,C,0,0,0,0,-,17
2,1.790988e+09,CMQdmOQ6dna2h8E97,192.168.150.1,50692,224.0.0.251,5353,udp,dns,-,-,...,T,T,0,D,1,73,0,0,-,17
3,1.790988e+09,C0tQa02W7XqnbcOjQd,192.168.150.1,53735,239.255.255.250,1900,udp,-,3.023189,680,...,T,T,0,D,4,792,0,0,-,17
4,1.790988e+09,CRuFYw3jKyOaRtErt7,192.168.150.129,50487,8.8.8.8,53,udp,dns,0.037343,0,...,T,F,0,Cd,0,0,1,422,-,17


In [19]:
target_ips = ['104.20.23.154', '172.66.147.243']
working_df = df[(df['conn_state'] == 'SHR')].sort_values(by='ts')


In [20]:
working_df['time_delta'] = working_df.groupby(["id.orig_h", "id.resp_h"])['ts'].diff()

In [21]:
stats = working_df.groupby(["id.orig_h", "id.resp_h"])['time_delta'].agg(['std','count'])
print(stats)

                                       std  count
id.orig_h       id.resp_h                        
192.168.150.129 104.20.23.154    40.366375      5
                172.66.147.243   53.842090      5
                192.168.150.254        NaN      0
                8.8.8.8          55.262958     19
                91.189.91.64           NaN      1
                91.189.91.98           NaN      0


In [23]:
working_df.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,time_delta
4,1.790988e+09,CRuFYw3jKyOaRtErt7,192.168.150.129,50487,8.8.8.8,53,udp,dns,0.037343,0,...,F,0,Cd,0,0,1,422,-,17,NaN
13,1.790988e+09,C73dne3j5u9NW6pzRk,192.168.150.129,53312,8.8.8.8,53,udp,dns,0.037651,0,...,F,0,Cd,0,0,1,278,-,17,89.984332
12,1.790988e+09,Cy3szQ1WGnEo6LX8x2,192.168.150.129,49340,91.189.91.98,80,tcp,-,0.066700,0,...,F,0,^hCadf,0,0,5,283,-,6,NaN
25,1.790988e+09,Ctv1Bc31oADVAxmfgk,192.168.150.129,54682,8.8.8.8,53,udp,dns,0.035377,0,...,F,0,Cd,0,0,1,422,-,17,210.066079
29,1.790988e+09,C4nhgl1zJFPzsztDO4,192.168.150.129,52889,8.8.8.8,53,udp,dns,0.061907,0,...,F,0,Cd,0,0,1,100,-,17,29.238044


In [25]:
print("mean gap:", working_df['time_delta'].mean())
print("std gap:", working_df['time_delta'].std())

mean gap: 50.05534460544586
std gap: 69.29044848134764


"Baseline with IP-grouping: beacon fragmented across two CDN IPs (std 40 / 53), indistinguishable from noise. Motivates domain-based grouping below."

In [26]:
dns_columns = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
               'proto', 'trans_id', 'rtt', 'query', 'qclass', 'qclass_name',
               'qtype', 'qtype_name', 'rcode', 'rcode_name', 'AA', 'TC', 'RD',
               'RA', 'Z', 'answers', 'TTLs', 'rejected', 'opcode', 'opcode_name']
df2 = pd.read_csv("../data/raw/zeek-captures/dns.log",sep='\t',comment='#', names = dns_columns)

df2[['query','answers']]

,query,answers
0,_ipps._tcp.local,-
1,connectivity-check.ubuntu.com,"2620:2d:4000:1::1099,2620:2d:4002:1::1062,2620..."
2,aliz_pc,-
3,aliz_pc,-
4,aliz_pc,-
5,aliz_pc,-
6,connectivity-check.ubuntu.com,"91.189.91.98,91.189.91.61,91.189.91.57,91.189...."
7,connectivity-check.ubuntu.com,"2620:2d:4002:1::1061,2620:2d:4002:1::1058,2620..."
8,example.com,"104.20.23.154,172.66.147.243"
9,example.com,"2606:4700:10::ac42:93f3,2606:4700:10::6814:179a"


In [27]:
# Step 1: keep only DNS rows that actually resolved to IPs (drop the "-" answers)
dns_resolved = df2[df2['answers'] != '-'].copy()

In [28]:
# Step 2: split the comma-joined answers string into a list, then explode to one row per IP
dns_resolved['answers'] = dns_resolved['answers'].str.split(',')
dns_exploded = dns_resolved.explode('answers')

In [29]:
# Step 3: keep only IPv4 answers (drop IPv6 addresses, which contain ':')
dns_exploded = dns_exploded[~dns_exploded['answers'].str.contains(':')]

In [30]:
# Step 4: build the IP -> domain lookup as a dictionary
ip_to_domain = dict(zip(dns_exploded['answers'], dns_exploded['query']))

In [31]:
# Step 5: add a dest_domain column to your conn.log dataframe
# (use whichever conn dataframe you're working with - the full deduped one)
df['dest_domain'] = df['id.resp_h'].map(ip_to_domain)

In [32]:
# connections whose IP had no DNS record get NaN; fill with the IP itself so they're still grouped
df['dest_domain'] = df['dest_domain'].fillna(df['id.resp_h'])

In [34]:
working = df[df['conn_state'] == 'SHR'].sort_values('ts')
working['time_delta'] = working.groupby(['id.orig_h', 'dest_domain'])['ts'].diff()

# per-pair summary: std (regularity) and count (how many connections)
stats_by_domain = working.groupby(['id.orig_h', 'dest_domain'])['time_delta'].agg(['std', 'count']).query('std < 2 and count >= 5')

print(stats_by_domain)

                                  std  count
id.orig_h       dest_domain                 
192.168.150.129 example.com  0.020402     11


Beacon detection via timing regularity (std of inter-arrival gaps). Naive IP-grouping fragmented the CDN-backed beacon across two Cloudflare IPs (std 40-53), hiding it among noise. Correlating conn.log with dns.log to group by destination domain unified the beacon, recovering its true regularity (std 0.02). Final detection requires both low std AND sufficient connection count — std alone false-positives on sparse regular traffic, count alone on high-volume irregular traffic.